# Stage 5F: Spectral-Floor Intervention

This notebook directly tests the mathematical hypothesis that the structural preconditioning explosion is fundamentally driven by the spectral floor $\lambda$.

Because the maximum possible amplification is bounded by:
$$ \|A^{-1/2}\|_2 \|B^{-1/2}\|_2 \lesssim \frac{1}{\lambda} $$

Varying $\lambda = 10^{-3}, 10^{-2}, 10^{-1}$ should proportionately shift the maximum observed `update_gain` ceiling down by orders of magnitude (from roughly $1000$ to $100$ to $10$).

In [ ]:
import json
import math
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
class InstrumentedSOAPStage5DE(optim.Optimizer):
    def __init__(
        self,
        params,
        lr: float = 1e-3,
        betas=(0.95, 0.95),
        damping: float = 1e-3,
        eps: float = 1e-8,
        precondition_frequency: int=10,
        update_mode: str = "direct_inverse_root",
    ):
        defaults = {
            "lr": lr,
            "betas": betas,
            "damping": damping,
            "eps": eps,
            "precondition_frequency": precondition_frequency,
            "update_mode": update_mode,
        }
        super().__init__(params, defaults)
        self.layer_diagnostics = []

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        self.layer_diagnostics = []
        
        for group in self.param_groups:
            update_mode = group["update_mode"]
            frequency = group["precondition_frequency"]
            damping = group["damping"]
            
            for layer_idx, p in enumerate(group["params"]):
                if p.grad is None:
                    continue
                
                grad = p.grad
                state = self.state[p]
                
                if "step" not in state:
                    state["step"] = 0
                
                if "exp_avg" not in state:
                    state["exp_avg"] = torch.zeros_like(grad)
                    state["exp_avg_sq"] = torch.zeros_like(grad)
                    
                if 'GG' not in state:
                    state['GG'] = []
                    if grad.dim() == 1:
                        state['GG'].append(torch.zeros(grad.shape[0], grad.shape[0], device=grad.device))
                    else:
                        for sh in grad.shape:
                            state['GG'].append(torch.zeros(sh, sh, device=grad.device))
                    
                    state['inv_sqrt'] = None
                    state['cached_eigens'] = None
                    
                    self._update_gg(grad, state, group['betas'][1])
                    state['inv_sqrt'], state['cached_eigens'] = self._get_inverse_root_and_eigens(state['GG'], damping)
                    
                    self.layer_diagnostics.append({"layer": layer_idx, "skipped": True})
                    continue
                
                state["step"] += 1

                inv_sqrt_fresh, eigens_fresh = self._get_inverse_root_and_eigens(state['GG'], damping)
                P_fresh = self._apply_inverse_root(grad, inv_sqrt_fresh)
                P_cached = self._apply_inverse_root(grad, state['inv_sqrt'])
                
                norm_P_fresh = float(torch.linalg.vector_norm(P_fresh))
                E_P = float(torch.linalg.vector_norm(P_cached - P_fresh)) / (norm_P_fresh + 1e-12)
                
                # Stage 5E/5F: Extract cached eigenvalue diagnostics
                stats_5e = {}
                if state.get('cached_eigens') is not None:
                    eigens = state['cached_eigens']
                    if len(eigens) > 0 and len(eigens[0]) > 0:
                        stats_5e['A_lmax'] = float(eigens[0].max())
                        stats_5e['A_lmin'] = float(eigens[0].min())
                        stats_5e['A_inv_norm2'] = 1.0 / math.sqrt(stats_5e['A_lmin'])
                    if len(eigens) > 1 and len(eigens[1]) > 0:
                        stats_5e['B_lmax'] = float(eigens[1].max())
                        stats_5e['B_lmin'] = float(eigens[1].min())
                        stats_5e['B_inv_norm2'] = 1.0 / math.sqrt(stats_5e['B_lmin'])
                
                if update_mode == "raw":
                    update_dir = grad
                elif update_mode == "raw_normalized":
                    exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                    beta1, beta2 = group["betas"]
                    
                    exp_avg.mul_(beta1).add_(grad, alpha=(1.0 - beta1))
                    exp_avg_sq.mul_(beta2).add_(grad.square(), alpha=(1.0 - beta2))
                    
                    denom = exp_avg_sq.sqrt().add_(group["eps"])
                    bias_correction1 = 1.0 - beta1 ** state["step"]
                    bias_correction2 = 1.0 - beta2 ** state["step"]
                    
                    update_dir = (exp_avg / denom) * (math.sqrt(bias_correction2) / bias_correction1)
                elif update_mode == "direct_inverse_root":
                    update_dir = P_cached
                elif update_mode == "direct_inverse_root_normalized":
                    exp_avg, exp_avg_sq = state["exp_avg"], state["exp_avg_sq"]
                    beta1, beta2 = group["betas"]
                    
                    exp_avg.mul_(beta1).add_(P_cached, alpha=(1.0 - beta1))
                    exp_avg_sq.mul_(beta2).add_(P_cached.square(), alpha=(1.0 - beta2))
                    
                    denom = exp_avg_sq.sqrt().add_(group["eps"])
                    bias_correction1 = 1.0 - beta1 ** state["step"]
                    bias_correction2 = 1.0 - beta2 ** state["step"]
                    
                    update_dir = (exp_avg / denom) * (math.sqrt(bias_correction2) / bias_correction1)
                else:
                    raise ValueError(f"Unknown update_mode: {update_mode}")
                    
                norm_G = float(torch.linalg.vector_norm(grad))
                delta_theta = update_dir * group["lr"]
                norm_update = float(torch.linalg.vector_norm(delta_theta))
                
                p.add_(update_dir, alpha=-group["lr"])
                
                diag = {
                    "layer": layer_idx,
                    "skipped": False,
                    "E_P": E_P,
                    "norm_G": norm_G,
                    "norm_update": norm_update,
                    "update_gain": norm_update / (group["lr"] * norm_G + 1e-12)
                }
                diag.update(stats_5e)
                self.layer_diagnostics.append(diag)
                
                self._update_gg(grad, state, group['betas'][1])
                
                if state['step'] > 0 and state['step'] % frequency == 0:
                    state['inv_sqrt'], state['cached_eigens'] = self._get_inverse_root_and_eigens(state['GG'], damping)
                    
        return loss

    def _update_gg(self, grad, state, beta2):
        if grad.dim() == 1:
            state['GG'][0].lerp_(grad.unsqueeze(1) @ grad.unsqueeze(0), 1 - beta2)
        else:
            state['GG'][0].lerp_(torch.tensordot(grad, grad, dims=[[1], [1]]), 1 - beta2)
            state['GG'][1].lerp_(torch.tensordot(grad, grad, dims=[[0], [0]]), 1 - beta2)

    def _get_inverse_root_and_eigens(self, mat_list, damping):
        inv_sqrts = []
        eigens = []
        for m in mat_list:
            if len(m) == 0:
                inv_sqrts.append([])
                eigens.append([])
                continue
            m64 = m.to(torch.float64)
            m64 = 0.5 * (m64 + m64.T)
            eigenvalues, eigenvectors = torch.linalg.eigh(m64 + damping*torch.eye(m.shape[0], device=m.device))
            eigenvalues = torch.clamp(eigenvalues, min=damping)
            inv_sqrt = torch.rsqrt(eigenvalues)
            res = eigenvectors @ torch.diag(inv_sqrt) @ eigenvectors.T
            inv_sqrts.append(res.to(m.dtype))
            eigens.append(eigenvalues.to(m.dtype))
        return inv_sqrts, eigens

    def _apply_inverse_root(self, grad, inv_sqrt_list):
        if grad.dim() == 1:
            if len(inv_sqrt_list[0]) > 0:
                return inv_sqrt_list[0] @ grad
            return grad
        else:
            res = grad
            if len(inv_sqrt_list[0]) > 0:
                res = inv_sqrt_list[0] @ res
            if len(inv_sqrt_list[1]) > 0:
                res = res @ inv_sqrt_list[1]
            return res


In [ ]:
class PINN(nn.Module):
    def __init__(self, width: int = 128, hidden_layers: int = 4) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy: torch.Tensor) -> torch.Tensor:
        return self.net(xy)

def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
def sample_collocation(n: int, device: torch.device) -> torch.Tensor:
    return torch.rand(n, 2, device=device, requires_grad=True)

def sample_boundary(n_each: int, device: torch.device) -> torch.Tensor:
    t = torch.rand(4 * n_each, device=device)
    x = torch.empty(4 * n_each, 2, device=device)
    x[:n_each] = torch.stack([torch.zeros(n_each, device=device), t[:n_each]], dim=1)
    x[n_each:2*n_each] = torch.stack([torch.ones(n_each, device=device), t[n_each:2*n_each]], dim=1)
    x[2*n_each:3*n_each] = torch.stack([t[2*n_each:3*n_each], torch.zeros(n_each, device=device)], dim=1)
    x[3*n_each:] = torch.stack([t[3*n_each:], torch.ones(n_each, device=device)], dim=1)
    return x

def make_batch_schedule(steps: int, K: int, n_pde: int, n_boundary_each: int, device: torch.device) -> List[Dict[str, torch.Tensor]]:
    schedule: List[Dict[str, torch.Tensor]] = []
    pde: Optional[torch.Tensor] = None
    bc: Optional[torch.Tensor] = None

    for step in range(1, steps + 1):
        if step == 1 or (step - 1) % K == 0:
            pde = sample_collocation(n_pde, device)
            bc = sample_boundary(n_boundary_each, device)
        schedule.append({
            "pde": pde.detach().clone(),
            "bc": bc.detach().clone(),
        })
    return schedule


In [ ]:
def poisson_residual(model: nn.Module, xy: torch.Tensor) -> torch.Tensor:
    u = model(xy)
    grad_u = torch.autograd.grad(u, xy, grad_outputs=torch.ones_like(u), create_graph=True, retain_graph=True)[0]
    u_x, u_y = grad_u[:, 0:1], grad_u[:, 1:2]
    u_xx = torch.autograd.grad(u_x, xy, grad_outputs=torch.ones_like(u_x), create_graph=True, retain_graph=True)[0][:, 0:1]
    u_yy = torch.autograd.grad(u_y, xy, grad_outputs=torch.ones_like(u_y), create_graph=True, retain_graph=True)[0][:, 1:2]
    forcing = 2.0 * math.pi**2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing

def relative_l2(model: nn.Module, device: torch.device, n: int = 81) -> float:
    grid = torch.linspace(0.0, 1.0, n, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing="ij")
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        pred = model(xy).reshape(-1)
    ref = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(pred - ref) / (torch.linalg.vector_norm(ref) + 1e-12))


In [ ]:
def run_stage5_f_audit(damping: float, K: int=1, steps: int=20, seed: int=0):
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = PINN().to(device)
    
    schedule_path = Path(f"poisson_seed{seed}_K{K}_schedule.pt")
    if schedule_path.exists():
        schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    else:
        schedule = make_batch_schedule(max(2000, steps), K, 1000, 100, device)
        torch.save(schedule, schedule_path)

    opt = InstrumentedSOAPStage5DE(
        model.parameters(), 
        lr=1e-3, 
        betas=(0.95, 0.95), 
        precondition_frequency=10, 
        update_mode="direct_inverse_root", 
        damping=damping
    )

    history = []
    model.train()
    for step in range(1, steps + 1):
        current_pde = schedule[step - 1]["pde"].to(device).detach().clone().requires_grad_(True)
        current_bc = schedule[step - 1]["bc"].to(device)
        opt.zero_grad(set_to_none=True)

        r = poisson_residual(model, current_pde)
        loss_pde = torch.mean(r.square())
        u_bc = model(current_bc)
        loss_bc = torch.mean(u_bc.square())
        loss = loss_pde + loss_bc
        loss.backward()

        opt.step()
        
        err = relative_l2(model, device)
        
        step_data = {
            "step": step,
            "loss": float(loss.detach()),
            "rel_l2": float(err),
            "layers": opt.layer_diagnostics
        }
        history.append(step_data)
        
        print(f"Step {step} | Loss {loss.item():.4e} | RelL2 {err:.4e}")

    return history


### Stage 5F: Testing the $\frac{1}{\lambda}$ Theoretical Ceiling

In [ ]:
print("Running K=1 Direct Inverse-Root with Damping 1e-3 (Ceiling ~ 1000)...")
audit_1e3 = run_stage5_f_audit(damping=1e-3, K=1, steps=20)
with open("stage5f_damping_1e-3.json", "w") as f:
    json.dump(audit_1e3, f, indent=2)


In [ ]:
print("\nRunning K=1 Direct Inverse-Root with Damping 1e-2 (Ceiling ~ 100)...")
audit_1e2 = run_stage5_f_audit(damping=1e-2, K=1, steps=20)
with open("stage5f_damping_1e-2.json", "w") as f:
    json.dump(audit_1e2, f, indent=2)


In [ ]:
print("\nRunning K=1 Direct Inverse-Root with Damping 1e-1 (Ceiling ~ 10)...")
audit_1e1 = run_stage5_f_audit(damping=1e-1, K=1, steps=20)
with open("stage5f_damping_1e-1.json", "w") as f:
    json.dump(audit_1e1, f, indent=2)
